# SeqEyes — Interactive Pulseq MRI Sequence Viewer

Drop‑in replacement for `pypulseq.Sequence.plot()`.

```python
import seqeyes
seqeyes.set(theme="dark")   # enable + set defaults

seq.plot()                           # interactive viewer inline
seq.plot(show_blocks=True)           # per‑call overrides
seq.plot(time_range=(0, 0.05))      # zoom to first 50 ms

seqeyes.reset()                      # back to matplotlib
```

**Plain `.py` script?**  `seq.plot()` opens your browser automatically.

In [1]:
import seqeyes
import numpy as np
import pypulseq as pp

# ── Enable SeqEyes (once per session) ──
seqeyes.set(time_disp="ms", grad_disp="Hz/m")
print(f"SeqEyes {seqeyes.__version__} — seq.plot() is now interactive!")

# ═══════════════════════════════════════════════════════════════════════
# 1. GRE  (from pypulseq/examples/scripts/write_gre.py)
# ═══════════════════════════════════════════════════════════════════════
fov_gre = 256e-3; n_x_gre = 64; n_y_gre = 64
flip_gre = 10; slice_thickness = 3e-3; tr_gre = 12e-3; te_gre = 5e-3

system_gre = pp.Opts(max_grad=28, grad_unit='mT/m', max_slew=150, slew_unit='T/m/s',
                      rf_ringdown_time=20e-6, rf_dead_time=100e-6, adc_dead_time=10e-6)
seq_gre = pp.Sequence(system_gre)

rf, gz, _ = pp.make_sinc_pulse(flip_angle=np.deg2rad(flip_gre), duration=3e-3,
                                slice_thickness=slice_thickness, apodization=0.42,
                                time_bw_product=4, system=system_gre, return_gz=True,
                                delay=system_gre.rf_dead_time, use='excitation')

delta_kx = 1 / fov_gre; delta_ky = 1 / fov_gre
gx = pp.make_trapezoid(channel='x', flat_area=n_x_gre * delta_kx, flat_time=3.2e-3, system=system_gre)
adc = pp.make_adc(num_samples=n_x_gre, duration=gx.flat_time, delay=gx.rise_time, system=system_gre)
gx_pre = pp.make_trapezoid(channel='x', area=-gx.area / 2, duration=1e-3, system=system_gre)
gz_reph = pp.make_trapezoid(channel='z', area=-gz.area / 2, duration=1e-3, system=system_gre)
gx_spoil = pp.make_trapezoid(channel='x', area=2 * n_x_gre * delta_kx, system=system_gre)
gz_spoil = pp.make_trapezoid(channel='z', area=4 / slice_thickness, system=system_gre)
phase_areas = (np.arange(n_y_gre) - n_y_gre / 2) * delta_ky

te_delay = te_gre - (pp.calc_duration(gz, rf) - pp.calc_rf_center(rf)[0] - rf.delay) - pp.calc_duration(gx_pre) - pp.calc_duration(gx) / 2 - pp.eps
te_delay = np.ceil(te_delay / seq_gre.grad_raster_time) * seq_gre.grad_raster_time
tr_delay = tr_gre - pp.calc_duration(gz, rf) - pp.calc_duration(gx_pre) - pp.calc_duration(gx) - te_delay
tr_delay = np.ceil(tr_delay / seq_gre.grad_raster_time) * seq_gre.grad_raster_time

rf_phase = 0; rf_inc = 0
for i_phase in range(n_y_gre):
    rf.phase_offset = rf_phase / 180 * np.pi
    adc.phase_offset = rf_phase / 180 * np.pi
    rf_inc = divmod(rf_inc + 117, 360.0)[1]
    rf_phase = divmod(rf_phase + rf_inc, 360.0)[1]
    seq_gre.add_block(rf, gz)
    gy_pre = pp.make_trapezoid(channel='y', area=phase_areas[i_phase], duration=pp.calc_duration(gx_pre), system=system_gre)
    seq_gre.add_block(gx_pre, gy_pre, gz_reph)
    seq_gre.add_block(pp.make_delay(te_delay))
    seq_gre.add_block(gx, adc)
    gy_pre.amplitude = -gy_pre.amplitude
    seq_gre.add_block(pp.make_delay(tr_delay), gx_spoil, gy_pre, gz_spoil)

# ═══════════════════════════════════════════════════════════════════════
# 2. EPI  (from pypulseq/examples/scripts/write_epi.py)
# ═══════════════════════════════════════════════════════════════════════
fov_epi = 220e-3; n_x_epi = 64; n_y_epi = 64; n_slices = 3

system_epi = pp.Opts(max_grad=32, grad_unit='mT/m', max_slew=130, slew_unit='T/m/s',
                      rf_ringdown_time=30e-6, rf_dead_time=100e-6)
seq_epi = pp.Sequence(system_epi)

rf_epi, gz_epi, _ = pp.make_sinc_pulse(flip_angle=np.pi / 2, system=system_epi, duration=3e-3,
                                        slice_thickness=slice_thickness, apodization=0.5,
                                        time_bw_product=4, return_gz=True,
                                        delay=system_epi.rf_dead_time, use='excitation')

dkx = 1 / fov_epi; dky = 1 / fov_epi; k_width = n_x_epi * dkx
adc_dwell = 4e-6; adc_duration = n_x_epi * adc_dwell
gx_flat = np.ceil(adc_duration * 1e5) * 1e-5
gx_epi = pp.make_trapezoid(channel='x', system=system_epi, amplitude=k_width / adc_duration, flat_time=gx_flat)
adc_epi = pp.make_adc(num_samples=n_x_epi, duration=adc_duration,
                       delay=gx_epi.rise_time + gx_flat / 2 - (adc_duration - adc_dwell) / 2)
pre_time = 8e-4
gx_pre_epi = pp.make_trapezoid(channel='x', system=system_epi, area=-gx_epi.area / 2, duration=pre_time)
gz_reph_epi = pp.make_trapezoid(channel='z', system=system_epi, area=-gz_epi.area / 2, duration=pre_time)
gy_pre_epi = pp.make_trapezoid(channel='y', system=system_epi, area=-n_y_epi / 2 * dky, duration=pre_time)
gy_blip_dur = np.ceil(2 * np.sqrt(dky / system_epi.max_slew) / 10e-6) * 10e-6
gy_epi = pp.make_trapezoid(channel='y', system=system_epi, area=dky, duration=gy_blip_dur)

for i_slice in range(n_slices):
    rf_epi.freq_offset = gz_epi.amplitude * slice_thickness * (i_slice - (n_slices - 1) / 2)
    seq_epi.add_block(rf_epi, gz_epi)
    seq_epi.add_block(gx_pre_epi, gy_pre_epi, gz_reph_epi)
    for _ in range(n_y_epi):
        seq_epi.add_block(gx_epi, adc_epi)
        seq_epi.add_block(gy_epi)
        gx_epi.amplitude = -gx_epi.amplitude

# ═══════════════════════════════════════════════════════════════════════
# 3. Radial GRE  (from pypulseq/examples/scripts/write_radial_gre.py)
# ═══════════════════════════════════════════════════════════════════════
fov_rad = 260e-3; n_x_rad = 64; flip_rad = 10; n_spokes = 60; n_dummy = 20
tr_rad = 20e-3; te_rad = 8e-3

system_rad = pp.Opts(max_grad=28, grad_unit='mT/m', max_slew=120, slew_unit='T/m/s',
                      rf_ringdown_time=20e-6, rf_dead_time=100e-6, adc_dead_time=10e-6)
seq_rad = pp.Sequence(system_rad)

rf_rad, gz_rad, _ = pp.make_sinc_pulse(apodization=0.5, duration=4e-3,
                                        flip_angle=np.deg2rad(flip_rad),
                                        slice_thickness=slice_thickness, system=system_rad,
                                        time_bw_product=4, return_gz=True,
                                        delay=system_rad.rf_dead_time, use='excitation')

dkx_rad = 1 / fov_rad
gx_rad = pp.make_trapezoid(channel='x', flat_area=n_x_rad * dkx_rad, flat_time=6.4e-3 / 5, system=system_rad)
adc_rad = pp.make_adc(num_samples=n_x_rad, duration=gx_rad.flat_time, delay=gx_rad.rise_time, system=system_rad)
gx_pre_rad = pp.make_trapezoid(channel='x', area=-gx_rad.area / 2 - dkx_rad / 2, duration=2e-3, system=system_rad)
gz_reph_rad = pp.make_trapezoid(channel='z', area=-gz_rad.area / 2, duration=2e-3, system=system_rad)
gx_spoil_rad = pp.make_trapezoid(channel='x', area=0.5 * n_x_rad * dkx_rad, system=system_rad)
gz_spoil_rad = pp.make_trapezoid(channel='z', area=4 / slice_thickness, system=system_rad)

te_delay_rad = te_rad - pp.calc_duration(gx_pre_rad) - gz_rad.fall_time - gz_rad.flat_time / 2 - pp.calc_duration(gx_rad) / 2
te_delay_rad = np.ceil(te_delay_rad / seq_rad.grad_raster_time) * seq_rad.grad_raster_time
tr_delay_rad = tr_rad - pp.calc_duration(gx_pre_rad) - pp.calc_duration(gz_rad) - pp.calc_duration(gx_rad) - te_delay_rad
tr_delay_rad = np.ceil(tr_delay_rad / seq_rad.grad_raster_time) * seq_rad.grad_raster_time

spoke_angle = np.pi / n_spokes
rf_phase = 0; rf_inc = 0
for i_spoke in range(-n_dummy, n_spokes + 1):
    rf_rad.phase_offset = rf_phase / 180 * np.pi
    adc_rad.phase_offset = rf_phase / 180 * np.pi
    rf_inc = divmod(rf_inc + 117, 360.0)[1]
    rf_phase = divmod(rf_inc + rf_phase, 360.0)[1]
    seq_rad.add_block(rf_rad, gz_rad)
    phi = spoke_angle * (i_spoke - 1)
    seq_rad.add_block(*pp.rotate(gx_pre_rad, gz_reph_rad, angle=phi, axis='z'))
    seq_rad.add_block(pp.make_delay(te_delay_rad))
    if i_spoke > 0:
        seq_rad.add_block(*pp.rotate(gx_rad, adc_rad, angle=phi, axis='z'))
    else:
        seq_rad.add_block(*pp.rotate(gx_rad, angle=phi, axis='z'))
    seq_rad.add_block(*pp.rotate(gx_spoil_rad, gz_spoil_rad, pp.make_delay(tr_delay_rad), angle=phi, axis='z'))

print(f"GRE: {len(seq_gre.block_events)} blocks  |  "
      f"EPI: {len(seq_epi.block_events)} blocks  |  "
      f"Radial: {len(seq_rad.block_events)} blocks")

d:\MIniconda\envs\NUM\Lib\site-packages\sigpy\config.py:27: UserWarning: Importing cupy.cuda.cudnn failed. For more details, see the error stack below:
DLL load failed while importing cudnn: The specified module could not be found.
  warnings.warn(


SeqEyes 0.1.0 — seq.plot() is now interactive!
GRE: 320 blocks  |  EPI: 390 blocks  |  Radial: 405 blocks


## 1. Gradient Echo — `write_gre.py`

Classic Cartesian GRE with sinc excitation, phase encoding, and
gradient spoiling.  64×64 matrix, TE=5 ms, TR=12 ms.

In [2]:
seq_gre.plot(time_range=(0, tr_gre), time_disp="ms", grad_disp="Hz/m", theme="dark")

d:\MIniconda\envs\NUM\Lib\site-packages\IPython\core\display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## 2. Echo Planar Imaging — `write_epi.py`

Single‑shot EPI with blipped phase encoding, alternating readout
polarity.  64×64, 3 slices, 220 mm FOV.

In [3]:
seq_epi.plot(time_disp="ms", grad_disp="Hz/m", theme="dark")

## 3. Radial GRE — `write_radial_gre.py`

Non‑Cartesian radial trajectory with 60 spokes, rotated readout
gradients, and RF spoiling.  Open the **K‑Space** panel!

In [4]:
seq_rad.plot(time_disp="ms", grad_disp="Hz/m", theme="dark")

## Customize the view

`seq.plot()` accepts the same arguments as pypulseq's original plot,
plus a `theme` keyword:

| Argument | Default | Description |
|---|---|---|
| `show_blocks` | `False` | Show block‑boundary lines |
| `time_range` | `(0, inf)` | Zoom to `(start_sec, end_sec)` |
| `time_disp` | `"s"` | Time unit: `"s"`, `"ms"`, `"us"` |
| `grad_disp` | `"Hz/m"` | Gradient unit: `"Hz/m"`, `"Hz/m"`, `"mT/m"`, `"G/cm"` |
| `theme` | `"system"` | `"light"`, `"dark"`, `"dracula"`, `"nord"`, ... |

In [5]:
# Zoom to first TR with block boundaries + custom units
seqeyes.set(show_blocks=True, time_disp="ms", grad_disp="Hz/m")
seq_gre.plot(time_range=(0, tr_gre), theme="dark")
seqeyes.reset()  # restore defaults

## Viewer Controls

| Action | How |
|---|---|
| **Zoom** | Scroll wheel |
| **Pan** | Click + drag |
| **Amplitude zoom (per channel)** | Ctrl + scroll wheel |
| **Tooltip** | Hover over waveform |
| **Toggle channels** | Click legend labels |
| **Block boundaries** | Check "Blocks" in toolbar |
| **K‑Space 3D viewer** | Click "K‑Space" button |
| **Rotate k‑space** | Click + drag in panel |
| **Change time unit** | Dropdown (s / ms / µs) |
| **Change gradient unit** | Dropdown (Hz/m / mT/m / G/cm) |
| **Change theme** | Dropdown in toolbar |
| **Open another .seq** | 📂 Open button |

## Plain `.py` script?

In a standalone Python script (no Jupyter), `seq.plot()` opens the
viewer in your default web browser.  See `examples/demo.py`.